In [ ]:
# importing packages

!pip install comet_ml > /dev/null 2>&1
import comet_ml
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
COMET_API_KEY = user_secrets.get_secret("COMET_API_KEY")

import torch
import torch.nn as nn
import torch.optim as optim

!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write
!apt-get install abcmidi timidity > /dev/null 2>&1

from IPython import display
import matplotlib.pyplot as plt

assert COMET_API_KEY != "", "Please insert your Comet API Key"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
songs = mdl.lab1.load_training_data()
songs_joined = "\n\n".join(songs)
vocab = sorted(set(songs_joined))

In [ ]:
# methods to pass from char to an id number and from an id number to char, we will use it to create a lookup table
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

In [ ]:
def vectorize_string(string):
    vector = np.array([char2idx[char] for char in string])
    return vector

vectorized_songs = vectorize_string(songs_joined)
assert isinstance(vectorized_songs, np.ndarray)

In [ ]:
def get_batch(vectorized_songs, seq_length, batch_size):
    # let's calculate the vectorized songs string max index to prevent OOB EX
    n = vectorized_songs.shape[0] - 1
    # let's choose a random idx for choose the batch start index
    idx = np.random.choice(n - seq_length, batch_size)
    # we take a number of input vector equals to the batch_size, and each one is traslated by one position
    input_batch = [vectorized_songs[i: i + seq_length] for i in idx]
    # the same logic of the output one, but it starts with the first traslated by one in the right, because is the prediction
    output_batch = [vectorized_songs[i+ 1: i + seq_length + 1] for i in idx]

    x_batch = torch.tensor(input_batch, dtype=torch.long).to(device)
    y_batch = torch.tensor(output_batch, dtype=torch.long).to(device)

    return x_batch, y_batch

test_args = (vectorized_songs, 2, 10)
x_batch, y_batch = get_batch(*test_args)
print(vectorized_songs.shape)
print(x_batch.shape)
print(y_batch.shape)
example_idx = 0
print("X (Input) :", x_batch[example_idx].tolist())
print("Y (Target):", y_batch[example_idx].tolist())

In [ ]:
# let's define our recurrent neural networ (RNN) model

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(LSTMModel, self).__init__()

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        # to convert the text in a vector of numbers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # LSTM analyze the input vectors and mantains an internal state, allowing the network to remember long time informations
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, dropout=0.2, batch_first=True)
        # receives in input the hidden_state and transforms it in the vector with the probability for each char
        self.fc = nn.Linear(hidden_size, vocab_size)

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
            torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):
        x = self.embedding(x)

        if state is None:
          state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)

        out = self.fc(out)
        return out if not return_state else (out, state)

In [ ]:
cross_entropy = nn.CrossEntropyLoss()
def compute_loss(labels, logits):
    batched_labels = labels.view(-1)
    batched_logits = logits.view(-1, logits.size(-1))
    loss = cross_entropy(batched_logits, batched_labels)
    return loss

In [ ]:
split = int(0.9 * len(vectorized_songs))
train_data = vectorized_songs[:split]
val_data = vectorized_songs[split:]

In [ ]:
def compute_val_loss(num_batches=20):
    model.eval()
    with torch.no_grad():
        losses = []
        for _ in range(num_batches):
            x, y = get_batch(val_data, params["seq_length"], params["batch_size"])
            y_hat = model(x)
            losses.append(compute_loss(y, y_hat).item())
    model.train()
    return np.mean(losses)

In [ ]:
def plot_losses(history, val_history):
    display.clear_output(wait=True)
    plt.figure(figsize=(10, 4))
    plt.plot(history, label="train loss")
    val_iters = [i * 10 for i in range(len(val_history))]
    plt.plot(val_iters, val_history, label="val loss")
    plt.xlabel("Iterations")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

In [ ]:
vocab_size = len(vocab)

params = dict(
    num_training_iteractions = 500,
    batch_size = 256,
    seq_length = 300,
    learning_rate = 5e-3,
    embedding_dim = 256,
    hidden_size = 1024,
    num_layers = 2
)

torch.cuda.empty_cache()

In [ ]:
def create_experiment():
  if 'experiment' in locals():
    experiment.end()

  experiment = comet_ml.Experiment(api_key=COMET_API_KEY, project_name="6S191_Lab1_Part2")

  for param, value in params.items():
    experiment.log_parameter(param, value)

  return experiment

In [ ]:
model = LSTMModel(vocab_size, params["embedding_dim"], params["hidden_size"], num_layers=params["num_layers"])
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

def train_step(x, y):
    model.train()
    optimizer.zero_grad()
    x = x.to(device)
    y = y.to(device)
    y_hat = model(x)
    loss = compute_loss(y, y_hat)
    loss.backward()
    optimizer.step()
    return loss

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
history = []
val_history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel="Iterations", ylabel="Loss")
experiment = create_experiment()

best_val_loss = float('inf')
patience = 5
no_improve = 0

if hasattr(tqdm, '_instances'): tqdm._instances.clear()
    
for iter in tqdm(range(params["num_training_iteractions"]), unit="it", leave=True):
    x_batch, y_batch = get_batch(train_data, params["seq_length"], params["batch_size"])

    loss = train_step(x_batch, y_batch)

    experiment.log_metric("loss", loss.item(), step=iter)

    history.append(loss.item())
    plot_losses(history, val_history)

    if iter % 10 == 0:
        val_loss = compute_val_loss()
        val_history.append(val_loss)
        experiment.log_metric("val_loss", val_loss, step=iter)
    
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_prefix)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at iter {iter} — best val_loss: {best_val_loss:.4f}")
                break

experiment.flush()
model.load_state_dict(torch.load(checkpoint_prefix))

In [ ]:
def generate_text(model, start_string, generation_length=1000):

  input_idx = [char2idx[s] for s in start_string]
  input_idx = torch.tensor([input_idx], dtype=torch.long).to(device)

  state = model.init_hidden(input_idx.size(0), device)

  text_generated = []
  tqdm._instances.clear()
  with torch.no_grad():
    for i in tqdm(range(generation_length)):
      predictions, state = model(input_idx, state, return_state=True)
      predictions = predictions.squeeze(0)

      input_idx = torch.multinomial(torch.softmax(predictions, dim=-1), num_samples=1)

      text_generated.append(idx2char[input_idx].item())

  return (start_string + ''.join(text_generated))

In [ ]:
generated_text = generate_text(model, start_string="X", generation_length=1000)
generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
  # Synthesize the waveform from a song
  waveform = mdl.lab1.play_song(song)

  # If its a valid song (correct syntax), lets play it!
  if waveform:
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # save your song to the Comet interface, you can access it there
    experiment.log_asset(wav_file_path)